# Milestone 5: Final Experiments and Analysis

This notebook compares the majority baseline, Logistic Regression, a class-weighted Decision Tree, and a class-weighted Random Forest across all five bankruptcy prediction horizons.



In [ ]:
# 1. Imports
from pathlib import Path
from datetime import datetime, timezone
import json
import platform
import subprocess
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score
)
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [ ]:
# 2. Set up paths and shared experiment settings
PROJECT_ROOT = Path.cwd()
DATA_ROOT = PROJECT_ROOT / "data" / "cleaned"
OUTPUT_DIR = PROJECT_ROOT / "results" / "milestone5"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
HORIZONS = ["1year", "2year", "3year", "4year", "5year"]
SELECTION_METRIC = "pr_auc"
BOOTSTRAP_SAMPLES = 1000

print("Project:", PROJECT_ROOT)
print("Output folder:", OUTPUT_DIR)
print("Horizons:", HORIZONS)

In [ ]:
# 2b. Define a practical public financial statement feature subset
# These ten ratios can be calculated from ordinary balance sheet and
# income statement.
PUBLIC_FINANCIAL_FEATURES = [
    "net profit / total assets",
    "total liabilities / total assets",
    "working capital / total assets",
    "current assets / short-term liabilities",
    "retained earnings / total assets",
    "EBIT / total assets",
    "sales / total assets",
    "equity / total assets",
    "gross profit / sales",
    "logarithm of total assets",
]

print(f"Public-financial-statement subset: {len(PUBLIC_FINANCIAL_FEATURES)} ratios")

In [ ]:
# 3. Helper functions for metrics and model creation
def get_metrics(y_true, predictions, probabilities):
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    return {
        "accuracy": accuracy_score(y_true, predictions),
        "precision": precision_score(y_true, predictions, zero_division=0),
        "recall": recall_score(y_true, predictions, zero_division=0),
        "f1": f1_score(y_true, predictions, zero_division=0),
        "roc_auc": roc_auc_score(y_true, probabilities),
        "pr_auc": average_precision_score(y_true, probabilities),
        "true_negatives": int(tn),
        "false_positives": int(fp),
        "false_negatives": int(fn),
        "true_positives": int(tp),
        "bankrupt_found": int(tp),
    }

def create_model(model_name, params):
    if model_name == "Majority baseline":
        return DummyClassifier(strategy="most_frequent")
    if model_name == "Logistic Regression":
        # Scaling is fit inside the pipeline using training data only.
        return make_pipeline(
            StandardScaler(),
            LogisticRegression(max_iter=2000, random_state=SEED, **params),
        )
    if model_name == "Class-weighted Decision Tree":
        # Trees do not need feature scaling.
        return DecisionTreeClassifier(
            class_weight="balanced", random_state=SEED, **params
        )
    if model_name == "Random Forest":
        return RandomForestClassifier(
            class_weight="balanced", random_state=SEED, n_jobs=-1, **params
        )
    raise ValueError(f"Unknown model: {model_name}")

In [ ]:
# 4. Define the small validation-set tuning grids
# Each trainable model has four pre-defined alternatives.

MODEL_GRIDS = {
    "Majority baseline": [{}],
    "Logistic Regression": [
        {"C": 0.1, "class_weight": None},
        {"C": 1.0, "class_weight": None},
        {"C": 0.1, "class_weight": "balanced"},
        {"C": 1.0, "class_weight": "balanced"},
    ],
    "Class-weighted Decision Tree": [
        {"max_depth": 3, "min_samples_leaf": 1},
        {"max_depth": 5, "min_samples_leaf": 1},
        {"max_depth": 7, "min_samples_leaf": 1},
        {"max_depth": 5, "min_samples_leaf": 5},
    ],
    "Random Forest": [
        {"n_estimators": 100, "max_depth": 5, "min_samples_leaf": 1},
        {"n_estimators": 100, "max_depth": 10, "min_samples_leaf": 1},
        {"n_estimators": 200, "max_depth": 5, "min_samples_leaf": 1},
        {"n_estimators": 200, "max_depth": 10, "min_samples_leaf": 1},
    ],
}
pd.DataFrame(
    [(model, json.dumps(params)) for model, grid in MODEL_GRIDS.items() for params in grid],
    columns=["Model", "Candidate parameters"],
)

In [ ]:
# 5. Tune on validation data only
# This cell records every candidate and keeps the strongest configuration
# per model and horizon by validation PR-AUC.
experiment_log_rows = []
selected_rows = []
selected_configs = {}

for horizon in HORIZONS:
    train = pd.read_csv(DATA_ROOT / horizon / f"{horizon}_train.csv")
    validation = pd.read_csv(DATA_ROOT / horizon / f"{horizon}_validation.csv")
    X_train, y_train = train.drop(columns="bankrupt"), train["bankrupt"]
    X_validation, y_validation = validation.drop(columns="bankrupt"), validation["bankrupt"]

    print(f"\nTuning {horizon}: {len(train):,} training rows, {len(validation):,} validation rows")
    for model_name, grid in MODEL_GRIDS.items():
        candidates = []
        for candidate_number, params in enumerate(grid, start=1):
            started = time.perf_counter()
            model = create_model(model_name, params)
            model.fit(X_train, y_train)
            probabilities = model.predict_proba(X_validation)[:, 1]
            predictions = model.predict(X_validation)
            scores = get_metrics(y_validation, predictions, probabilities)
            elapsed = time.perf_counter() - started
            candidates.append((params, scores))
            experiment_log_rows.append({
                "run_id": f"{horizon}-{model_name.lower().replace(' ', '-')}-val-{candidate_number}",
                "stage": "validation_tuning",
                "horizon": horizon,
                "model": model_name,
                "parameters": json.dumps(params, sort_keys=True),
                "selection_metric": SELECTION_METRIC,
                "seed": SEED,
                "split_used": "training + validation; test not used",
                **scores,
                "runtime_seconds": round(elapsed, 2),
                "status": "completed",
                "notes": "Candidate selected only by validation PR-AUC.",
            })

        # If PR-AUC ties, prefer fewer false positives.
        best_params, best_scores = max(
            candidates, key=lambda item: (item[1]["pr_auc"], -item[1]["false_positives"])
        )
        selected_configs[(horizon, model_name)] = best_params
        selected_rows.append({
            "horizon": horizon,
            "model": model_name,
            "selected_parameters": json.dumps(best_params, sort_keys=True),
            "validation_pr_auc": best_scores["pr_auc"],
            "validation_recall": best_scores["recall"],
            "validation_false_positives": best_scores["false_positives"],
        })

selected_configurations = pd.DataFrame(selected_rows)
selected_configurations.round(3)

## Final test evaluation

The next cell retrains each selected configuration using the combined training and validation data, then evaluates it once on the reserved test data. Do not alter selected configurations based on these results.

In [ ]:
# 6. Final test evaluation after configuration selection
final_rows = []
error_rows = []
test_probabilities_by_model = {}

for horizon in HORIZONS:
    train = pd.read_csv(DATA_ROOT / horizon / f"{horizon}_train.csv")
    validation = pd.read_csv(DATA_ROOT / horizon / f"{horizon}_validation.csv")
    test = pd.read_csv(DATA_ROOT / horizon / f"{horizon}_test.csv")
    X_fit = pd.concat([train.drop(columns="bankrupt"), validation.drop(columns="bankrupt")], ignore_index=True)
    y_fit = pd.concat([train["bankrupt"], validation["bankrupt"]], ignore_index=True)
    X_test, y_test = test.drop(columns="bankrupt"), test["bankrupt"]

    for model_name in MODEL_GRIDS:
        params = selected_configs[(horizon, model_name)]
        started = time.perf_counter()
        model = create_model(model_name, params)
        model.fit(X_fit, y_fit)
        probabilities = model.predict_proba(X_test)[:, 1]
        predictions = model.predict(X_test)
        scores = get_metrics(y_test, predictions, probabilities)
        elapsed = time.perf_counter() - started
        final_rows.append({
            "horizon": horizon,
            "model": model_name,
            "selected_parameters": json.dumps(params, sort_keys=True),
            **scores,
        })
        experiment_log_rows.append({
            "run_id": f"{horizon}-{model_name.lower().replace(' ', '-')}-final-test",
            "stage": "final_test",
            "horizon": horizon,
            "model": model_name,
            "parameters": json.dumps(params, sort_keys=True),
            "selection_metric": SELECTION_METRIC,
            "seed": SEED,
            "split_used": "training + validation fit; test evaluated once after selection",
            **scores,
            "runtime_seconds": round(elapsed, 2),
            "status": "completed",
            "notes": "Final test result; not used to choose configuration.",
        })
        test_probabilities_by_model[(horizon, model_name)] = (y_test.to_numpy(), probabilities)

        examples = pd.DataFrame({
            "test_row_index": X_test.index,
            "actual_class": y_test.to_numpy(),
            "predicted_class": predictions,
            "predicted_bankruptcy_probability": probabilities,
        })
        for error_type, mask in [
            ("false_negative", (examples.actual_class == 1) & (examples.predicted_class == 0)),
            ("false_positive", (examples.actual_class == 0) & (examples.predicted_class == 1)),
        ]:
            if mask.any():
                error_rows.append({
                    "horizon": horizon,
                    "model": model_name,
                    "error_type": error_type,
                    **examples.loc[mask].iloc[0].to_dict(),
                })

main_test_results = pd.DataFrame(final_rows)
main_test_results[[
    "horizon", "model", "accuracy", "precision", "recall", "f1",
    "roc_auc", "pr_auc", "false_positives", "false_negatives"
]].round(3)

In [ ]:
# 7. Feature ablation: all 64 ratios versus 10 practical ratios
# This uses validation data.
# The selected Random Forest settings are held fixed so the input features
# are the only difference.
ablation_rows = []
for horizon in HORIZONS:
    train = pd.read_csv(DATA_ROOT / horizon / f"{horizon}_train.csv")
    validation = pd.read_csv(DATA_ROOT / horizon / f"{horizon}_validation.csv")
    X_train, y_train = train.drop(columns="bankrupt"), train["bankrupt"]
    X_validation, y_validation = validation.drop(columns="bankrupt"), validation["bankrupt"]
    params = selected_configs[(horizon, "Random Forest")]

    for input_set, features in [
        ("All 64 financial ratios", list(X_train.columns)),
        ("10 core public-financial-statement ratios", PUBLIC_FINANCIAL_FEATURES),
    ]:
        model = create_model("Random Forest", params)
        model.fit(X_train[features], y_train)
        probabilities = model.predict_proba(X_validation[features])[:, 1]
        predictions = model.predict(X_validation[features])
        ablation_rows.append({
            "horizon": horizon,
            "model": "Random Forest",
            "input_set": input_set,
            "feature_count": len(features),
            "parameters": json.dumps(params, sort_keys=True),
            **get_metrics(y_validation, predictions, probabilities),
            "split_used": "validation only; final test set not used",
        })

public_feature_ablation = pd.DataFrame(ablation_rows)
display(public_feature_ablation[[
    "horizon", "input_set", "feature_count", "pr_auc", "recall"
]].round(3))

fig, axes = plt.subplots(1, 2, figsize=(10.5, 4.4), constrained_layout=True)
labels = ["All 64 ratios", "10 core ratios"]
colors = ["#059669", "#7c3aed"]
x = np.arange(len(HORIZONS))
for axis, metric, title in zip(
    axes, ["pr_auc", "recall"], ["Validation PR-AUC", "Validation recall"]
):
    for index, (input_set, label, color) in enumerate(zip(public_feature_ablation["input_set"].unique(), labels, colors)):
        subset = public_feature_ablation[public_feature_ablation["input_set"] == input_set].set_index("horizon").reindex(HORIZONS)
        axis.bar(x + (index - 0.5) * 0.35, subset[metric], 0.35, label=label, color=color)
    axis.set_title(title, fontweight="bold")
    axis.set_xticks(x, [item.replace("year", " year") for item in HORIZONS])
    axis.set_ylim(0, 1.05)
    axis.set_ylabel("Score", fontweight="bold")
    axis.grid(axis="y", alpha=0.25)
axes[0].legend(frameon=False)
fig.suptitle("Feature ablation: full ratios versus a practical 10-ratio subset", fontweight="bold")
fig.savefig(OUTPUT_DIR / "public_feature_ablation_figure.png", dpi=240, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# 7. Estimate uncertainty with stratified bootstrap 95% confidence intervals
# Resamples final test predictions within each class.
def bootstrap_intervals(y_true, probabilities, rng):
    negative_indices = np.flatnonzero(y_true == 0)
    positive_indices = np.flatnonzero(y_true == 1)
    samples = []
    for _ in range(BOOTSTRAP_SAMPLES):
        sampled_indices = np.concatenate([
            rng.choice(negative_indices, size=len(negative_indices), replace=True),
            rng.choice(positive_indices, size=len(positive_indices), replace=True),
        ])
        sample_y = y_true[sampled_indices]
        sample_probabilities = probabilities[sampled_indices]
        sample_predictions = (sample_probabilities >= 0.5).astype(int)
        samples.append(get_metrics(sample_y, sample_predictions, sample_probabilities))
    sample_frame = pd.DataFrame(samples)
    return {
        metric: (
            sample_frame[metric].quantile(0.025),
            sample_frame[metric].quantile(0.975),
        )
        for metric in ["precision", "recall", "f1", "roc_auc", "pr_auc"]
    }

rng = np.random.default_rng(SEED)
ci_rows = []
for (horizon, model_name), (y_test, probabilities) in test_probabilities_by_model.items():
    point_estimates = main_test_results[
        (main_test_results.horizon == horizon) & (main_test_results.model == model_name)
    ].iloc[0]
    for metric, (lower, upper) in bootstrap_intervals(y_test, probabilities, rng).items():
        ci_rows.append({
            "horizon": horizon,
            "model": model_name,
            "metric": metric,
            "estimate": point_estimates[metric],
            "ci_95_lower": lower,
            "ci_95_upper": upper,
            "bootstrap_samples": BOOTSTRAP_SAMPLES,
        })

bootstrap_confidence_intervals = pd.DataFrame(ci_rows)
bootstrap_confidence_intervals.round(3).head(12)

In [ ]:
# 8. Create the main report figure: PR-AUC and recall by horizon
model_order = [
    "Majority baseline", "Logistic Regression",
    "Class-weighted Decision Tree", "Random Forest"
]
colors = ["#64748b", "#2563eb", "#d97706", "#059669"]
x = np.arange(len(HORIZONS))
width = 0.19

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5), constrained_layout=True)
for axis, metric, title in zip(
    axes,
    ["pr_auc", "recall"],
    ["Test PR-AUC", "Test recall (bankrupt class)"],
):
    for index, (model_name, color) in enumerate(zip(model_order, colors)):
        subset = main_test_results[main_test_results.model == model_name].set_index("horizon").reindex(HORIZONS)
        values = subset[metric].to_numpy()
        ci_subset = bootstrap_confidence_intervals[
            (bootstrap_confidence_intervals.model == model_name) &
            (bootstrap_confidence_intervals.metric == metric)
        ].set_index("horizon").reindex(HORIZONS)
        yerr = np.vstack([
            values - ci_subset["ci_95_lower"].to_numpy(),
            ci_subset["ci_95_upper"].to_numpy() - values,
        ])
        axis.bar(
            x + (index - 1.5) * width, values, width, label=model_name, color=color,
            yerr=yerr, capsize=3, error_kw={"ecolor": "#1f2937", "elinewidth": 0.8},
        )
    axis.set_title(title, fontweight="bold")
    axis.set_xticks(x, [name.replace("year", " year") for name in HORIZONS])
    axis.set_ylim(0, 1.05)
    axis.set_ylabel("Score", fontweight="bold")
    axis.grid(axis="y", alpha=0.25)

axes[0].legend(loc="upper left", frameon=False, fontsize=9)
fig.suptitle(
    "Final test-set performance across bankruptcy prediction horizons\n"
    "Error bars show stratified-bootstrap 95% confidence intervals",
    fontsize=15, fontweight="bold",
)
fig.savefig(OUTPUT_DIR / "main_results_figure.png", dpi=240, bbox_inches="tight", facecolor="white")
plt.show()

In [ ]:
# 9. Review errors and Decision Tree depth sensitivity
error_examples = pd.DataFrame(error_rows)
tree_depth_sensitivity = pd.DataFrame(experiment_log_rows).query(
    "stage == 'validation_tuning' and model == 'Class-weighted Decision Tree'"
)

display(error_examples.head(12))
display(tree_depth_sensitivity[[
    "horizon", "parameters", "pr_auc", "recall", "false_positives", "false_negatives"
]].round(3))

In [ ]:
# 10. Save the results package and experiment log
main_test_results.to_csv(OUTPUT_DIR / "main_test_results.csv", index=False)
selected_configurations.to_csv(OUTPUT_DIR / "selected_configurations.csv", index=False)
pd.DataFrame(experiment_log_rows).to_csv(OUTPUT_DIR / "experiment_log.csv", index=False)
bootstrap_confidence_intervals.to_csv(OUTPUT_DIR / "bootstrap_confidence_intervals.csv", index=False)
error_examples.to_csv(OUTPUT_DIR / "error_examples.csv", index=False)
tree_depth_sensitivity.to_csv(OUTPUT_DIR / "tree_depth_sensitivity.csv", index=False)
public_feature_ablation.to_csv(OUTPUT_DIR / "public_feature_ablation.csv", index=False)

run_metadata = {
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "random_seed": SEED,
    "selection_metric": SELECTION_METRIC,
    "bootstrap_samples": BOOTSTRAP_SAMPLES,
    "horizons": HORIZONS,
    "python": sys.version,
    "scikit_learn": sklearn.__version__,
    "platform": platform.platform(),
    "git_commit": subprocess.getoutput("git rev-parse HEAD"),
}
(OUTPUT_DIR / "run_metadata.json").write_text(json.dumps(run_metadata, indent=2), encoding="utf-8")

print("Saved:")
for file in sorted(OUTPUT_DIR.iterdir()):
    print("-", file.name)